In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-22")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7b2f48c2-cd6b-4df8-b4ba-8b77d8fc9249;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 146ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

26/08/07 11:34:28 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


**Problem 1 | Easy**

Orders above a threshold, sorted
Find all orders with unit_price greater than 300 AND status equal to "Delivered". Return order_id, customer_id, unit_price, sorted by unit_price descending.

In [ ]:
from pyspark.sql import functions as F
filtered_orders=(orders_df.filter((F.col('unit_price')>300) & (F.col('status')=='Delivered'))).\
    select('order_id','customer_id','unit_price').\
    orderBy(F.col('unit_price').desc())
filtered_orders.show(truncate=False)

+--------+-----------+----------+
|order_id|customer_id|unit_price|
+--------+-----------+----------+
|O0001   |C001       |1299.99   |
|O0009   |C009       |1299.99   |
|O0024   |C004       |1299.99   |
|O0034   |C014       |1299.99   |
|O0041   |C021       |1299.99   |
|O0051   |C006       |1299.99   |
|O0088   |C018       |1299.99   |
|O0094   |C024       |1299.99   |
|O0015   |C015       |699.99    |
|O0037   |C017       |699.99    |
|O0056   |C011       |699.99    |
|O0098   |C003       |699.99    |
|O0012   |C012       |599.99    |
|O0031   |C011       |599.99    |
|O0055   |C010       |599.99    |
|O0002   |C002       |449.99    |
|O0020   |C020       |449.99    |
|O0029   |C009       |449.99    |
|O0042   |C022       |449.99    |
|O0089   |C019       |449.99    |
+--------+-----------+----------+
only showing top 20 rows


**Problem 2 | Easy**

Revenue and order count by region
For each region, calculate the total revenue (unit_price * quantity) and the number of orders. Sort by total revenue descending.

In [14]:
agg = (
    orders_df
    .groupBy("region")
    .agg(
        F.round(
            F.sum(F.col("unit_price") * F.col("quantity")), 2
        ).alias("total_revenue"),
        F.count("order_id").alias("total_orders")
    )
    .orderBy(F.col("total_revenue").desc())
)

agg.show(truncate=False)

+-------+-------------+------------+
|region |total_revenue|total_orders|
+-------+-------------+------------+
|West   |15054.3      |36          |
|East   |12119.32     |27          |
|Midwest|9369.5       |16          |
|South  |7839.56      |21          |
+-------+-------------+------------+



**Problem 3 | Medium**

Regions with more than N orders
Find all regions that have placed more than 20 orders. Show region and order_count. This requires filtering on an aggregated value — the SQL HAVING equivalent.

In [19]:
(orders_df.groupBy('region')
 .agg(F.count('order_id')
      .alias('order_count'))
      .filter(F.col('order_count')>20)
      .orderBy(F.col('order_count')
      .desc())).show(truncate=False)

+------+-----------+
|region|order_count|
+------+-----------+
|West  |36         |
|East  |27         |
|South |21         |
+------+-----------+



**Problem 4 | Medium**

Average order value by payment method, excluding cancelled
Excluding any order with status = "Cancelled", calculate the average unit_price grouped by payment_method. Round to 2 decimal places. Sort descending.

In [20]:
(orders_df.
 filter(F.col('status')!='Cancelled')
 .groupBy('payment_method')
 .agg(F.round(F.avg('unit_price'),2).alias('avg_unit_price'))
 .orderBy(F.col('avg_unit_price').desc())).show()

+--------------+--------------+
|payment_method|avg_unit_price|
+--------------+--------------+
|    Debit Card|        377.32|
|   Credit Card|        330.31|
|        PayPal|         259.2|
+--------------+--------------+



**Problem 5 | Hard**

Customers with above-average spend

Find all customers whose total spend (sum of unit_price across all their orders) is greater than the overall average total spend per customer across the entire dataset. This requires computing an aggregate, then comparing each customer's aggregate against it.

In [ ]:
customer_spend = (
    orders_df
    .groupBy("customer_id")
    .agg(F.sum("unit_price").alias("total_spend"))
)

avg_spend = (
    customer_spend
    .agg(F.avg("total_spend"))
    .collect()[0][0]
)

result = (
    customer_spend
    .filter(F.col("total_spend") > avg_spend)
    .orderBy(F.col("total_spend").desc())
)
result.show(10)

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|       C001|    2299.95|
|       C010|    2179.96|
|       C002|    1979.95|
|       C009|    1899.96|
|       C006|    1879.96|
|       C024|    1799.97|
|       C014|    1769.96|
|       C018|    1754.96|
|       C004|    1609.95|
|       C016|    1529.96|
+-----------+-----------+
only showing top 10 rows


**Problem 6 | Hard**

Multi-level grouping with conditional aggregation
For each region, calculate: total order count, count of Delivered orders only, and the percentage of orders that were delivered (rounded to 1 decimal). Use conditional aggregation — SUM(CASE WHEN ... THEN 1 ELSE 0 END) in SQL, F.sum(F.when(...)) in PySpark.

In [34]:
(orders_df.groupBy("region")
 .agg(F.count("order_id").alias("total_orders"),
F.sum(F.when(F.col("status") == "Delivered", 1).otherwise(0)).alias("delivered_orders")
).withColumn(
    "pct_delivered",
    F.round(100.0 * F.col("delivered_orders") / F.col("total_orders"), 1)
).orderBy(F.col("pct_delivered").desc())).show()

+-------+------------+----------------+-------------+
| region|total_orders|delivered_orders|pct_delivered|
+-------+------------+----------------+-------------+
|  South|          21|              16|         76.2|
|   West|          36|              27|         75.0|
|   East|          27|              20|         74.1|
|Midwest|          16|              11|         68.8|
+-------+------------+----------------+-------------+

